In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [2]:
#Load keypoint's coordinates already extracted

train_dataset = pd.read_csv("hand_dataset_train.csv")
X = train_dataset.iloc[:, 1:].values
Y = train_dataset.iloc[:, 0].values

# split train-validation
X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)


test_dataset = pd.read_csv("hand_dataset_test.csv")
X_test = test_dataset.iloc[:, 1:].values
y_test = test_dataset.iloc[:, 0].values

In [3]:
#standardize keypoints coordinates

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("Observations and features train:", X_train.shape)
print("Observations and features validation:", X_val.shape)
print("Observations and features test:", X_test.shape)

Observations and features train: (21576, 42)
Observations and features validation: (5394, 42)
Observations and features test: (2952, 42)


In [8]:
#See if the classes are balanced or not

classes = np.unique(y_train)
for cls in classes:
    X_cls_train = X_train[y_train == cls]
    print(f"Classe {cls}: {X_cls_train.shape[0]} samples")


Classe A: 713 samples
Classe B: 507 samples
Classe C: 938 samples
Classe D: 926 samples
Classe E: 806 samples
Classe F: 1023 samples
Classe G: 986 samples
Classe H: 966 samples
Classe I: 966 samples
Classe J: 370 samples
Classe K: 1154 samples
Classe L: 1021 samples
Classe M: 732 samples
Classe N: 724 samples
Classe O: 703 samples
Classe P: 841 samples
Classe Q: 697 samples
Classe R: 1022 samples
Classe S: 903 samples
Classe T: 774 samples
Classe U: 694 samples
Classe V: 990 samples
Classe W: 923 samples
Classe X: 970 samples
Classe Y: 860 samples
Classe Z: 367 samples


In [9]:
#Because we are using a generative classifier with unbalanced classes we should consider also priors

priors = {}
for cls in classes:
    priors[cls] = np.sum(y_train == cls) / len(y_train)

In [13]:
# We should choose the number of gaussian components for each letter. This choice is based on 3 different loss functions
component_candidates = [1, 2, 3, 4, 5, 6, 8, 10]

#Initialize cìthe dictionaries
best_components_bic = {}
best_components_aic = {}
best_components_nll  = {}
best_gmms_bic = {}
best_gmms_aic = {}
best_gmms_nll  = {}

for cls in classes:
    Xc_train = X_train[y_train == cls] #number of component should be chosen for each class
    Xc_val   = X_val[y_val == cls] #and on validation set

    best_bic = np.inf
    best_aic = np.inf
    best_nll  = np.inf
    best_gmm_bic = None
    best_gmm_aic = None
    best_gmm_nll  = None

    for k in component_candidates:
        gmm = GaussianMixture(n_components=k,covariance_type='full',random_state=42)

        gmm.fit(Xc_train)

        # three different criteria used: negative log-likelihood, AIC and BIC
        nll  = - gmm.score(Xc_val) * len(Xc_val)
        aic = gmm.aic(Xc_val)
        bic = gmm.bic(Xc_val)
        
        # Update the best negative log likelihood and the best k
        if nll < best_nll:
            best_nll = nll
            best_components_nll[cls] = k
            best_gmms_nll[cls] = gmm

        # Update the best AIC and the best k
        if aic < best_aic:
            best_aic = aic
            best_components_aic[cls] = k
            best_gmms_aic[cls] = gmm

        # Update the best BIC and the best k
        if bic < best_bic:
            best_bic = bic
            best_components_bic[cls] = k
            best_gmms_bic[cls] = gmm

    print(f"Classe {cls}: NLL={best_components_nll[cls]}, AIC={best_components_aic[cls]}, BIC={best_components_bic[cls]}")


Classe A: NLL=3, AIC=2, BIC=1
Classe B: NLL=2, AIC=2, BIC=1
Classe C: NLL=3, AIC=2, BIC=1
Classe D: NLL=2, AIC=2, BIC=1
Classe E: NLL=4, AIC=2, BIC=1
Classe F: NLL=4, AIC=2, BIC=1
Classe G: NLL=5, AIC=2, BIC=1
Classe H: NLL=2, AIC=2, BIC=1
Classe I: NLL=3, AIC=3, BIC=1
Classe J: NLL=1, AIC=1, BIC=1
Classe K: NLL=5, AIC=3, BIC=1
Classe L: NLL=4, AIC=2, BIC=1
Classe M: NLL=3, AIC=2, BIC=1
Classe N: NLL=3, AIC=2, BIC=1
Classe O: NLL=3, AIC=2, BIC=1
Classe P: NLL=2, AIC=2, BIC=1
Classe Q: NLL=2, AIC=2, BIC=1
Classe R: NLL=2, AIC=2, BIC=1
Classe S: NLL=5, AIC=2, BIC=1
Classe T: NLL=3, AIC=2, BIC=1
Classe U: NLL=3, AIC=2, BIC=1
Classe V: NLL=3, AIC=3, BIC=1
Classe W: NLL=4, AIC=2, BIC=1
Classe X: NLL=3, AIC=3, BIC=1
Classe Y: NLL=2, AIC=1, BIC=1
Classe Z: NLL=2, AIC=1, BIC=1


In [14]:
#Re-train on train and validation set

X_train_full = np.concatenate([X_train, X_val], axis=0)
y_train_full = np.concatenate([y_train, y_val], axis=0)

final_gmms_nll  = {}
final_gmms_aic = {}
final_gmms_bic = {}

for cls in classes:

    k_bic = best_components_bic[cls]
    k_aic = best_components_aic[cls]
    k_nll  = best_components_nll[cls]

    Xc = X_train_full[y_train_full == cls]

    #NLL
    print(f"Train for class {cls} with {k_nll} components (NLL)")
    gmm_nll = GaussianMixture(n_components=k_nll,covariance_type='full',random_state=42)
    gmm_nll.fit(Xc)
    final_gmms_nll[cls] = gmm_nll

    #AIC
    print(f"Train for class {cls} with {k_aic} components (AIC)")
    gmm_aic = GaussianMixture(n_components=k_aic,covariance_type='full',random_state=42)
    gmm_aic.fit(Xc)
    final_gmms_aic[cls] = gmm_aic

    #BIC
    print(f"Train for class {cls} with {k_bic} components (BIC)")
    gmm_bic = GaussianMixture(n_components=k_bic,covariance_type='full',random_state=42)
    gmm_bic.fit(Xc)
    final_gmms_bic[cls] = gmm_bic

Train for class A with 3 components (NLL)
Train for class A with 2 components (AIC)
Train for class A with 1 components (BIC)
Train for class B with 2 components (NLL)
Train for class B with 2 components (AIC)
Train for class B with 1 components (BIC)
Train for class C with 3 components (NLL)
Train for class C with 2 components (AIC)
Train for class C with 1 components (BIC)
Train for class D with 2 components (NLL)
Train for class D with 2 components (AIC)
Train for class D with 1 components (BIC)
Train for class E with 4 components (NLL)
Train for class E with 2 components (AIC)
Train for class E with 1 components (BIC)
Train for class F with 4 components (NLL)
Train for class F with 2 components (AIC)
Train for class F with 1 components (BIC)
Train for class G with 5 components (NLL)
Train for class G with 2 components (AIC)
Train for class G with 1 components (BIC)
Train for class H with 2 components (NLL)
Train for class H with 2 components (AIC)
Train for class H with 1 component

In [20]:
#Evaluate on test set

def predict_gmms(X, gmms, priors, classes):
    y_pred = []
    for x in X:
        class_scores = {}
        for cls in classes:
            log_like = gmms[cls].score(x.reshape(1, -1))
            log_post = log_like + np.log(priors[cls])
            nll = -log_post
            class_scores[cls] = nll
        #choose the class with lowest NLL
        y_pred.append(min(class_scores, key=class_scores.get))
    return np.array(y_pred)

# Prediction NLL
y_pred_nll = predict_gmms(X_test, final_gmms_nll, priors, classes)
acc_nll = accuracy_score(y_test, y_pred_nll)
print(f"Test accuracy (NLL): {acc_nll*100:.2f}%")

# Prediction AIC
y_pred_aic = predict_gmms(X_test, final_gmms_aic, priors, classes)
acc_aic = accuracy_score(y_test, y_pred_aic)
print(f"Test accuracy (AIC): {acc_aic*100:.2f}%")

# Prediction BIC
y_pred_bic = predict_gmms(X_test, final_gmms_bic, priors, classes)
acc_bic = accuracy_score(y_test, y_pred_bic)
print(f"test accuracy (BIC): {acc_bic*100:.2f}%")

Test accuracy (NLL): 77.30%
Test accuracy (AIC): 84.08%
test accuracy (BIC): 77.57%


In [21]:
print("\nClassification report (NLL):")
print(classification_report(y_test, y_pred_nll, digits=4))


Classification report (NLL):
              precision    recall  f1-score   support

           A     1.0000    0.7826    0.8780       115
           B     1.0000    1.0000    1.0000       115
           C     1.0000    0.9904    0.9952       104
           D     0.9905    0.9043    0.9455       115
           E     0.9914    1.0000    0.9957       115
           F     1.0000    1.0000    1.0000       115
           G     0.0000    0.0000    0.0000       115
           H     1.0000    1.0000    1.0000       115
           I     0.4852    1.0000    0.6534       115
           J     1.0000    0.0087    0.0172       115
           K     1.0000    0.9043    0.9498       115
           L     1.0000    0.6174    0.7634       115
           M     0.3732    0.4609    0.4125       115
           N     0.7261    1.0000    0.8413       114
           O     1.0000    0.2809    0.4386        89
           P     0.4936    1.0000    0.6609       115
           Q     0.7042    0.8696    0.7782       1

In [22]:
print("\nClassification report (AIC):")
print(classification_report(y_test, y_pred_aic, digits=4))


Classification report (AIC):
              precision    recall  f1-score   support

           A     0.8295    0.9304    0.8770       115
           B     1.0000    1.0000    1.0000       115
           C     1.0000    0.9904    0.9952       104
           D     0.9865    0.6348    0.7725       115
           E     0.8129    0.9826    0.8898       115
           F     1.0000    1.0000    1.0000       115
           G     0.0000    0.0000    0.0000       115
           H     1.0000    1.0000    1.0000       115
           I     0.4936    1.0000    0.6609       115
           J     1.0000    0.0087    0.0172       115
           K     0.9914    1.0000    0.9957       115
           L     1.0000    0.8609    0.9252       115
           M     0.8958    0.3739    0.5276       115
           N     1.0000    1.0000    1.0000       114
           O     1.0000    1.0000    1.0000        89
           P     0.5204    1.0000    0.6845       115
           Q     0.7353    0.8696    0.7968       1

c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [23]:
print("Classification report (BIC):")
print(classification_report(y_test, y_pred_bic, digits=4))

Classification report (BIC):
              precision    recall  f1-score   support

           A     0.9340    0.8609    0.8959       115
           B     1.0000    1.0000    1.0000       115
           C     1.0000    1.0000    1.0000       104
           D     1.0000    0.6174    0.7634       115
           E     0.9426    1.0000    0.9705       115
           F     1.0000    1.0000    1.0000       115
           G     0.0000    0.0000    0.0000       115
           H     1.0000    1.0000    1.0000       115
           I     0.4457    1.0000    0.6166       115
           J     0.0000    0.0000    0.0000       115
           K     0.9914    1.0000    0.9957       115
           L     1.0000    0.6174    0.7634       115
           M     0.0000    0.0000    0.0000       115
           N     0.6867    1.0000    0.8143       114
           O     1.0000    0.5955    0.7465        89
           P     0.4661    0.9565    0.6268       115
           Q     0.4231    0.9565    0.5867       11

c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\.venv11\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [25]:
#Conformal prediction adapted to GMM: Mondrian conformal prediction

#Split test set in calibration and final test
X_calib, X_test_final, y_calib, y_test_final = train_test_split(X_test, y_test, test_size=0.5, random_state=42, stratify=y_test)
print(f"Calibration set: {X_calib.shape[0]} samples")
print(f"Final test set: {X_test_final.shape[0]} samples\n")

# Define significance level, alpha
alpha = 0.1

# Nonconformity score is defined as the negative log likelihood under the true class that consider also the prior because the class are unbalanced
nonconformity_scores_calib = {cls: [] for cls in classes}

for i, x in enumerate(X_calib):
    true_class = y_calib[i]
    gmm = final_gmms_aic[true_class]
    
    # Log-likelihood under the GMM of the true class
    log_like = gmm.score(x.reshape(1, -1))
    
    # Nonconformity score
    log_prior = np.log(priors[true_class])
    nonconf_score = -(log_like + log_prior)
    
    nonconformity_scores_calib[true_class].append(nonconf_score)

Calibration set: 1476 samples
Final test set: 1476 samples



In [26]:
#Compute quantiles, one for each class

quantiles = {}
for cls in classes:
    scores = np.array(nonconformity_scores_calib[cls])
    n_cls = len(scores)

    q_level = np.ceil((n_cls + 1) * (1 - alpha)) / n_cls #Conformal quantile formula
    q_level = min(q_level, 1.0)  #max it could be one
    quantiles[cls] = np.quantile(scores, q_level) #Choose the quantile for the specific class based on the calibration scores
    print(f"Class {cls}: n={n_cls}, quantile={quantiles[cls]:.4f}")

Class A: n=58, quantile=54.2446
Class B: n=58, quantile=-51.3911
Class C: n=52, quantile=139.3890
Class D: n=58, quantile=19.5675
Class E: n=57, quantile=-1.2122
Class F: n=57, quantile=72.9434
Class G: n=58, quantile=813.8554
Class H: n=57, quantile=172.0324
Class I: n=57, quantile=-22.7677
Class J: n=58, quantile=236.3214
Class K: n=58, quantile=28.8633
Class L: n=57, quantile=198.7072
Class M: n=58, quantile=245.4122
Class N: n=57, quantile=40.3770
Class O: n=45, quantile=107.6026
Class P: n=58, quantile=36.7317
Class Q: n=58, quantile=117.7066
Class R: n=57, quantile=32.4764
Class S: n=58, quantile=-6.4956
Class T: n=57, quantile=66.4775
Class U: n=58, quantile=-13.2689
Class V: n=57, quantile=5.0985
Class W: n=57, quantile=-23.9726
Class X: n=57, quantile=198.5795
Class Y: n=57, quantile=15.5573
Class Z: n=57, quantile=110.6153


In [32]:
#Prediction sets for the final test

def predict_conformal_mondrian(X, gmms, classes, quantiles):
    predictions = []
    prediction_sets = []
    
    for x in X:
        pred_set = []
        nonconf_scores = {}
        
        # Compute nonconformity score for each class
        for cls in classes:
            log_like = gmms[cls].score(x.reshape(1, -1))
            nonconf_score = -log_like
            nonconf_scores[cls] = nonconf_score
            
            # Include class in prediction set if score ≤ class-specific quantile
            if nonconf_score <= quantiles[cls]:
                pred_set.append(cls)
        
        if len(pred_set) == 0:
            pred_set = [min(nonconf_scores, key=nonconf_scores.get)]
        
        prediction_sets.append(pred_set)
        
        # Punctual prediction: class with minimum nonconformity (min NLL considering also prior)
        predictions.append(min(nonconf_scores, key=nonconf_scores.get))
    
    return np.array(predictions), prediction_sets

# Make predictions
y_pred_conf, pred_sets = predict_conformal_mondrian(X_test_final, final_gmms_aic, classes, quantiles)

#Evaluation

# Coverage (proportion of times true class is in prediction set)
coverage_list = [1 if y_test_final[i] in pred_sets[i] else 0 
                 for i in range(len(y_test_final))]
avg_coverage = np.mean(coverage_list)
print(f"\n COVERAGE:")
print(f"   Average Coverage: {avg_coverage*100:.2f}%")

# Cardinality (size of prediction sets)
cardinalities = [len(pred_set) for pred_set in pred_sets]
avg_cardinality = np.mean(cardinalities)
print(f"\n CARDINALITY:")
print(f"   Average Set Size: {avg_cardinality:.2f}")
print(f"   Min Set Size: {min(cardinalities)}")
print(f"   Max Set Size: {max(cardinalities)}")

# Per-class coverage
print(f"\n PER-CLASS COVERAGE:")
class_coverage = {}
for cls in classes:
    mask = y_test_final == cls
    if np.sum(mask) > 0:
        cls_coverage = np.mean([1 if cls in pred_sets[i] else 0 
                                for i in range(len(y_test_final)) if mask[i]])
        class_coverage[cls] = cls_coverage
        print(f"   Class {cls}: {cls_coverage*100:.1f}% (n={np.sum(mask)})")


 COVERAGE:
   Average Coverage: 94.24%

 CARDINALITY:
   Average Set Size: 2.34
   Min Set Size: 1
   Max Set Size: 6

 PER-CLASS COVERAGE:
   Class A: 86.0% (n=57)
   Class B: 87.7% (n=57)
   Class C: 98.1% (n=52)
   Class D: 96.5% (n=57)
   Class E: 98.3% (n=58)
   Class F: 100.0% (n=58)
   Class G: 93.0% (n=57)
   Class H: 100.0% (n=58)
   Class I: 91.4% (n=58)
   Class J: 94.7% (n=57)
   Class K: 100.0% (n=57)
   Class L: 96.6% (n=58)
   Class M: 94.7% (n=57)
   Class N: 87.7% (n=57)
   Class O: 100.0% (n=44)
   Class P: 100.0% (n=57)
   Class Q: 89.5% (n=57)
   Class R: 84.5% (n=58)
   Class S: 89.5% (n=57)
   Class T: 98.3% (n=58)
   Class U: 96.5% (n=57)
   Class V: 89.7% (n=58)
   Class W: 100.0% (n=58)
   Class X: 100.0% (n=58)
   Class Y: 94.8% (n=58)
   Class Z: 84.5% (n=58)


In [33]:
print("EXAMPLES OF PREDICTION SETS")

n_examples = min(40, len(X_test_final))
for i in range(n_examples):
    true_class = y_test_final[i]
    pred_set = pred_sets[i]
    point_pred = y_pred_conf[i]
    in_set = "✓" if true_class in pred_sets[i] else "✗"
    correct = "✓" if point_pred == true_class else "✗"
    
    print(f"Sample {i+1:3d}: True={true_class} | Pred={point_pred} {correct} | "
          f"Set={sorted(pred_set)} | Size={len(pred_set)} | Covered={in_set}")

EXAMPLES OF PREDICTION SETS
Sample   1: True=T | Pred=T ✓ | Set=['T', 'X'] | Size=2 | Covered=✓
Sample   2: True=V | Pred=U ✗ | Set=['H', 'U'] | Size=2 | Covered=✗
Sample   3: True=Y | Pred=Y ✓ | Set=['J', 'Y'] | Size=2 | Covered=✓
Sample   4: True=R | Pred=R ✓ | Set=['H'] | Size=1 | Covered=✗
Sample   5: True=P | Pred=P ✓ | Set=['P', 'Q'] | Size=2 | Covered=✓
Sample   6: True=W | Pred=W ✓ | Set=['W'] | Size=1 | Covered=✓
Sample   7: True=H | Pred=H ✓ | Set=['H'] | Size=1 | Covered=✓
Sample   8: True=K | Pred=K ✓ | Set=['K'] | Size=1 | Covered=✓
Sample   9: True=W | Pred=W ✓ | Set=['W'] | Size=1 | Covered=✓
Sample  10: True=Y | Pred=Y ✓ | Set=['J', 'Y'] | Size=2 | Covered=✓
Sample  11: True=J | Pred=I ✗ | Set=['J'] | Size=1 | Covered=✓
Sample  12: True=O | Pred=O ✓ | Set=['O'] | Size=1 | Covered=✓
Sample  13: True=B | Pred=B ✓ | Set=['B', 'C', 'M'] | Size=3 | Covered=✓
Sample  14: True=C | Pred=C ✓ | Set=['C', 'M'] | Size=2 | Covered=✓
Sample  15: True=N | Pred=N ✓ | Set=['G', 'J', 'M'

In [35]:
# Distribution of set sizes
print("DISTRIBUTION OF PREDICTION SET SIZES")
from collections import Counter
size_dist = Counter(cardinalities)
for size in sorted(size_dist.keys()):
    count = size_dist[size]
    pct = count / len(cardinalities) * 100
    print(f"Size {size:2d}: {count:4d} samples ({pct:5.1f}%)")

DISTRIBUTION OF PREDICTION SET SIZES
Size  1:  539 samples ( 36.5%)
Size  2:  401 samples ( 27.2%)
Size  3:  203 samples ( 13.8%)
Size  4:  190 samples ( 12.9%)
Size  5:  116 samples (  7.9%)
Size  6:   27 samples (  1.8%)
